# Projection-based transport morphometry (simulation)

Compare regularized logistic classification using raw pixels, sliced intensity-mass quantiles, or their concatenation. Construct 24 quantiles along each of 12 angles. Use stratified 85% training subsamples and compare all arms on the same test images. Each seed generates 1,800 training and 600 held-out images with class-dependent shape and texture. Intensities form synthetic spatial distributions.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'projection-transport-morphometry-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom scipy import ndimage as ndi\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import balanced_accuracy_score, roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", str(ROOT / "data")))\nSEEDS = [13, 37, 61]\n\n\ndef classifier():\n    return make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=700))\n\n\ndef save(model, name):\n    path = ROOT / "checkpoints"\n    path.mkdir(exist_ok=True)\n    joblib.dump(model, path / (name + ".joblib"), compress=3)\n\n\ndef phantom(n, seed):\n    rng = np.random.default_rng(seed)\n    y = rng.integers(2, size=n)\n    u, v = np.mgrid[-1:1:28j, -1:1:28j]\n    x = []\n    for label in y:\n        a, b = rng.normal(0, 0.22, 2)\n        scale = rng.uniform(0.14, 0.38)\n        field = np.exp(\n            -(((u - a) / (scale * (1 + 0.55 * label))) ** 2 + ((v - b) / scale) ** 2)\n            / 2\n        )\n        grain = ndi.gaussian_filter(rng.normal(size=u.shape), 0.5 + 0.4 * label)\n        x.append(np.clip(0.30 + field * rng.uniform(0.20, 0.5) + grain * 0.16, 0, 1))\n    return np.asarray(x), y\n\n\ndef image_data(seed, filename):\n    return [phantom(n, seed + offset) for n, offset in [(1800, 0), (600, 100)]]\n\n\ndef evaluate(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "balanced_accuracy": float(balanced_accuracy_score(y, p >= 0.5)),\n    }\n\n\ndef transport(x):\n    projections = []\n    for angle in range(0, 180, 15):\n        rotated = ndi.rotate(\n            x, angle, axes=(1, 2), reshape=True, order=1, prefilter=False\n        )\n        mass = np.maximum(rotated, 0).sum(axis=1) + 1e-8\n        cdf = np.cumsum(mass, axis=1) / mass.sum(1)[:, None]\n        coord = np.linspace(-1, 1, cdf.shape[1])\n        q = np.linspace(0.03, 0.97, 24)\n        projections.append(np.stack([np.interp(q, row, coord) for row in cdf]))\n    return np.hstack(projections)\n\n\ndef run(seed):\n    (x, y), (z, t) = image_data(seed, "pneumoniamnist.npz")\n    tr, _ = train_test_split(\n        np.arange(len(y)), train_size=0.85, stratify=y, random_state=seed\n    )\n    a = transport(x[tr])\n    b = transport(z)\n    out = {}\n    for name, f, g in [\n        ("pixels", x[tr].reshape(len(tr), -1), z.reshape(len(z), -1)),\n        ("transport", a, b),\n        (\n            "combined",\n            np.c_[a, x[tr].reshape(len(tr), -1)],\n            np.c_[b, z.reshape(len(z), -1)],\n        ),\n    ]:\n        model = classifier().fit(f, y[tr])\n        save(model, f"{name}-{seed}")\n        out[name] = evaluate(t, model.predict_proba(g)[:, 1])\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())